## Generating summary & bias for each article:

In [ ]:
!git clone https://github.com/probcomp/hfppl.git
!cd hfppl && pip install . && cd ..

In [1]:
import sys
sys.path.append("/data/cb/scratch/bfefferm/NLP-Project/hfppl")

In [2]:
import pandas as pd
import csv
import os
from hfppl import Model, LMContext, TokenCategorical, CachedCausalLM, smc_steer, smc_standard
from score import compute_pbf_score, compute_pbi_score
from smc_steer_summary import bias_model_factory, TwistModel, gen_summary

**Loading Dataset:**

In [3]:
dataset = pd.read_csv('../POLITICS_finetuning/processed_data.csv')

In [4]:
dataset

,title,body,stance
0,Elizabeth Cheney blasted by older sister over ...,"Call it Cheney versus Cheney.\nMary Cheney, on...",center
1,Mary Cheney: Sister Is 'Dead Wrong' On Gay Mar...,"Mary Cheney, the younger sister of Wyoming U.S...",center
2,IRS official who refused to testify facing mor...,The IRS official who refused to testify at a H...,center
3,White House Plays Down Data Program,WASHINGTON — The Obama administration tried Sa...,center
4,N.R.A. Details Plan for Armed School Guards,Report Sees Guns as Path to Safety in Schools\...,center
...,...,...,...
295,Nancy Pelosi Re-Elected House Minority Leader,WASHINGTON ― House Minority Leader Nancy Pelos...,right
296,Nancy Pelosi Beats Back House Democratic Leade...,WASHINGTON — House Democrats on Wednesday reje...,center
297,Obama Will Meet With Sanders On Thursday,WASHINGTON -- With presumptive Democratic pres...,left
298,"Clinton Is 'Sane' And 'Competent,' Unlike Trum...",PHILADELPHIA ― Americans should vote for Hilla...,center


In [5]:
# need to login to use llama
from huggingface_hub import notebook_login

notebook_login()

**Specifying file paths:**

In [6]:
from transformers import AutoTokenizer

In [7]:
# Model paths:
import torch

print('Loading bias model...')
bias_model_path = '/data/cb/scratch/bfefferm/NLP-Project-storage/Saved_Models/politics_best_2500/politics_best_2500_30bz_000007_best'
bias_tokenizer_path = '/data/cb/scratch/bfefferm/NLP-Project-storage/Saved_Models/politics_best_2500/tokenizer_politics_best_2500_30bz_000007_best'

bias_model = bias_model_factory(bias_model_path, bias_tokenizer_path)

print('Loaded!')


# Specifying model name:
# llm_model_name = 'gpt2'

# from transformers import GPT2Tokenizer, GPT2LMHeadModel

# gpt2_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
# gpt2_model = GPT2LMHeadModel.from_pretrained('gpt2',  load_in_8bit=True, device_map='auto')

from transformers import LlamaForCausalLM, LlamaTokenizer

llama_tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-7b-hf", 
                                                torch_dtype=torch.float32,
                                               cache_dir='/data/cb/scratch/bfefferm/NLP-Project-storage/Saved_Models/hf_cache')
llama_model = LlamaForCausalLM.from_pretrained("meta-llama/Llama-2-7b-hf", 
                                               torch_dtype=torch.float32, 
                                               device_map='auto',
                                              cache_dir='/data/cb/scratch/bfefferm/NLP-Project-storage/Saved_Models/hf_cache')
llama_tokenizer.model_max_length = 1024

llm = CachedCausalLM(llama_model, llama_tokenizer)


# Loading llm:
# llm = CachedCausalLM.from_pretrained(llm_model_name, load_in_8bit=True)

Loading bias model...
Loaded!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

**Iterating over each summary, predicting bias, & saving to `.csv`:**

In [8]:
num_summaries = 3

In [ ]:
# Open file  
with open('../POLITICS_finetuning/processed_data.csv') as file_obj: 
    # Create reader object by passing the file  
    # object to reader method 
    reader_obj = csv.reader(file_obj) 
    with open('../summaries/llama-smc.csv', 'w') as f:  
        # Initialize writer object:
        writer_obj = csv.writer(f)
        # The fields of this file are
        # ['title', 'body', 'stance']
        # Iterate over each row in the .csv,
        # skipping the first row (pertaining to field / column)
        next(reader_obj)
        writer_obj.writerow(['Title', 'Summary', 'Predicted Bias', 'Stance'])
        
    for row in reader_obj: 
        # Store title:
        title = row[0]
        # Store article:
        article = row[1]
        # Ensuring that articles fit within maximum length
        # For each stance:
        for stance in ['left', 'center', 'right']:
            for i in range(num_summaries):
                summary = await gen_summary('llama', llm, bias_model, TwistModel, article, stance)
                # For each summary, predict its bias:
                # pred_bias, logits = bias_model(summary)
                with open('../summaries/llama-smc.csv', 'a') as f: 
                    # Initialize writer object:
                    writer_obj = csv.writer(f)
                    writer_obj.writerow([title, summary, None, stance])
        

next sentence: Power to the people.
1.7578179
next sentence: Call it Cheney versus Cheney.
-0.27789378
next sentence: The family feud between Elizabeth Cheney and her lesbian sister.
-0.6920675
next sentence: All families are created equal and deserve equal rights and protections.
-1.2365221
next sentence: The older Cheneys, radio dispatched (or drawn and quartered, if you will?
0.48660862
next sentence: Liz Cheney is taking aim at Mary’s support of same-sex marriage, splitting apart her family and setting off a controversy.
-1.5295141
next sentence: Mary Cheney, sister of Wyoming Senate candidate Elizabeth Cheney, calls for marriage equality after Liz’s negative stance on the issue.
-4.4802704
next sentence: I wish I was there for a 1-on-1 with the two ladies about whether family separation is an actual punishment or just a burden of love.
-3.6974277
next sentence: Mary Cheney says Liz Cheney is wrong on gay marriage, and her ongoing conflict with DJT
Liz Cheney is betraying her conse